# Exercise: Can an encoder find malaria?

Each image is one blood cell. Some cells are **parasitized** (infected with malaria). Some are **uninfected**.

We used a pretrained encoder (DINOv2) to turn each image into a row of 768 numbers.

## Setup

In [ ]:
library('tidymodels')
library('embed')

options(repr.plot.res = 250, repr.plot.height = 3, repr.plot.width = 5)

## 1. Load the data

Here are 6 example cells. The top row is parasitized. The bottom row is uninfected.

<img src="https://raw.githubusercontent.com/UNC-BIOS-512/jupyterlite/main/data/malaria_examples/parasitized_C129P90ThinF_IMG_20151004_133921_cell_105.png" width="90"><img src="https://raw.githubusercontent.com/UNC-BIOS-512/jupyterlite/main/data/malaria_examples/parasitized_C189P150ThinF_IMG_20151203_140809_cell_133.png" width="90"> <img src="https://raw.githubusercontent.com/UNC-BIOS-512/jupyterlite/main/data/malaria_examples/parasitized_C52P13thinF_IMG_20150725_130013_cell_194.png" width="90"> <br><img src="https://raw.githubusercontent.com/UNC-BIOS-512/jupyterlite/main/data/malaria_examples/uninfected_C147P108ThinF_IMG_20151115_101156_cell_82.png" width="90"> <img src="https://raw.githubusercontent.com/UNC-BIOS-512/jupyterlite/main/data/malaria_examples/uninfected_C184P145ThinF_IMG_20151203_102405_cell_137.png" width="90"> <img src="https://raw.githubusercontent.com/UNC-BIOS-512/jupyterlite/main/data/malaria_examples/uninfected_C77P38ThinF_IMG_20150601_152941_cell_55.png" width="90">

The encoder already ran. Each row of `malaria` is one image. `e1` to `e768` are its embedding.

In [ ]:
malaria <- readr::read_csv('https://raw.githubusercontent.com/UNC-BIOS-512/jupyterlite/main/data/malaria_embeddings.csv.gz', show_col_types = FALSE) |>
    mutate(label = factor(label, levels = c('uninfected', 'parasitized')))

malaria |> select(id, label, e1:e5) |> head()

In [ ]:
malaria |> count(label)

## 2. Visualize with UMAP

We cannot plot 768 dimensions at once. **UMAP** squeezes all the dimensions into 2 new columns, `UMAP1` and `UMAP2`. Similar images wil have similar `UMAP1` and `UMAP2` values.

Run the cell. It makes the 2 UMAP columns. (It takes a few seconds.)

In [ ]:
set.seed(1)

umap_coords <- recipe(label ~ ., data = select(malaria, -id)) |>
    step_umap(all_predictors(), num_comp = 2) |>
    prep() |>
    bake(new_data = NULL)

umap_coords |> head()

**Your turn:** plot `UMAP1` against `UMAP2`. Color the points by `label`.

In [ ]:
# Chart code goes here

**Check:** do the two kinds of cell land in different places?

- Do you see clear groups? Are they perfectly separate?
- Which cells are hard to tell apart?

## 3. Split the data

Run this cell. Do not change it.

- `train`: half of the images. The model learns from these.
- `test`: the other half. The model has never seen these. We measure accuracy here.

In [ ]:
set.seed(512)

split <- initial_split(malaria, prop = 0.5, strata = label)
train <- training(split) |> select(-id)
test  <- testing(split)  |> select(-id)

nrow(train)

## 4. Pick a model

Pick a classifier model that we've worked with before (e.g. random forest). Fit it on `train`. Find the accuracy on `test`.

Options:

- `logistic_reg(penalty = 0.01, mixture = 0) |> set_engine('glmnet')`: logistic regression. We have 768 columns, so it needs a `penalty` (like in lecture 16).
- `rand_forest(trees = 200) |> set_mode('classification') |> set_engine('ranger')`: random forest.
- `mlp(hidden_units = 10, epochs = 500) |> set_mode('classification') |> set_engine('nnet', MaxNWts = 10000)`: neural net.

Use the formula `label ~ .` (all other embeddings columns are predictors).

In [ ]:
# Fit your model here

<details>
<summary>Hint: how do I find the accuracy?</summary>

1. `predict()` gives the predicted class for each test image.
2. `bind_cols()` puts it next to the true `label`.
3. `accuracy()` compares them.

```r
predict(fit, new_data = test) |>
    bind_cols(test) |>
    accuracy(truth = label, estimate = .pred_class)
```

Here `fit` is the name of your fitted model. Use your own name.

</details>

In [ ]:
# Accuracy on the test set

**Check:** a good model gets about 0.93 to 0.95 on the test images.

- Why did you pick this model?
- Is the accuracy on `train` higher than on `test`? What does that tell you?

## 5. Bonus: raw pixels

We also have the raw pixels for the same cells: `malaria_pixels.csv.gz`. That is 32 x 32 = 1,024 columns named `p1` to `p1024`.

Use the same split. Fit the same kind of model on the pixels. How does the test accuracy compare to the embeddings?

In [ ]:
pixels <- readr::read_csv('https://raw.githubusercontent.com/UNC-BIOS-512/jupyterlite/main/data/malaria_pixels.csv.gz', show_col_types = FALSE) |>
    mutate(label = factor(label, levels = c('uninfected', 'parasitized')))

# Use the same split: train_pix has the same images as train
train_pix <- pixels |> filter(id %in% training(split)$id) |> select(-id)
test_pix  <- pixels |> filter(id %in% testing(split)$id)  |> select(-id)

# Fit and evaluate here

*Your answer:* Which is better for finding malaria: pixels or embeddings? Why?